## Reflection Groups in Action: Algorithms for Orbits, Cosets and Vectors

Götz Pfeiffer - University of Galway

# 1. Algorithms: A Glimpse of CGT

![Trees](images/trees.jpg)

## Algorithms

A good algorithm is short yet powerful.

Like this one-line recursive implementation of the [**Euclidean Algorithm**](https://en.wikipedia.org/wiki/Euclidean_algorithm):

In [ ]:
gcd(a, b) =  b == 0 ?  a  :  gcd(b, a % b)

In [ ]:
gcd(237934564903214447864796765423105634076804062380125, 46805092789549878742860339647868490386740234760875)

This code is directly translated from the corresponding mathematical formula:
$$
\gcd(a, b) = \begin{cases}
a, & \text{if } b = 0,\\
\gcd(b, a \bmod b), & \text{else,}
\end{cases}
$$
which is helpful for **proving desirable properties** of an algorithm (meaning, termination, correctness, ...).  Also it can serve as a **model for variations** (for speed, ...) or **extensions** (progress reports, additional data, ...).

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* A **simple function** can be defined by just assigning a formula to the function call expression, like
  ```julia
  square(x) = x^2
  ```
* Then `square(2)` should return `4`.
* The **ternary operator** `c ? a : b`, as in `gcd` above, has the value `a` if the condition `c` is true, and `b` otherwise ([conditional evaluation](https://docs.julialang.org/en/v1/manual/control-flow/#man-conditional-evaluation)).

</div>

Many of my algorithms are variations or extensions of the **orbit algorithm**, a simple $3$-step procedure of the form:

1. **Initialize.** (a list)
2. **Loop.**  (over the list and grow it)
3. **Return.**  (the list)

We will soon see that the list really is a **tree**, and that the algorithm performs some kind of **tree traversal**.

##  Tree Traversal: BFS vs DFS

* A **tree** is a connected, cycle-free (simple) graph.  A **rooted tree** is a tree with a distinguished node, the **root** of the tree.
* With respect to the root, each (non-root) node in the tree has a unique **parent** node, of which it is a **child** node.
* Implicitly, a rooted tree is then a **digraph** (i.e., a directed graph) with arrows pointing away from the root, from the parent to its children.

<div class="alert alert-block alert-success">

* **Example.** Here is a tree with node set $\{1,2, \ldots, 7\}$ and root $6$ (in white):

![e7 tree](images/e7tree.png)
![e7 direct](images/e7direct.png)

The parent of node $2$ is node $4$.  The children of node $4$ are nodes $2$ and $3$.
</div>

* **Breadth First Search** (BFS) and **Depth First Search**  (DFS) provide strategies for traversing a tree in a simple and systematic fashion.
* For this, we assume that each **node** $x$ in the tree knows all its **children** as a list $x$.next.
* BFS uses a FIFO **queue** (first in, first out), while DFS uses a LIFO **stack** (last in, first out).
* We use `pop()` to remove an item, and `push()` to add items to either data structure.
* Then the two **algorithms** for visiting all nodes of a tree with root $x$ can be described as an initialization step, followed by a loop.

![bfs vs dfs](images/bfsdfs.png)

* In practice, BFS can take advantage of **dynamic lists** in a `for` loop.
* DFS, using the **function call stack**, can be implemented **recursively**.

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* A **for loop** has the general form
  ```julia
  for <var> in <list>
      <body>
  end
  ```
* A **function definition** has the general form
  ```julia
  function <name> ( <params> )
      <body>
  end
  ```

</div>

Here is an implementation of BFS in `julia`, with a `visit` function as an explicit argument:

In [ ]:
function BFS(x, visit)
    Q = [x]
    for y in Q
        visit(y)
        append!(Q, y.next)
    end
end

And here is DFS, again with a `visit` function as an explicit argument:

In [ ]:
function DFS(x, visit)
    visit(x)
    for z in x.next
        DFS(z, visit)
    end
end

To apply this to the example tree above, we organize the tree as a collection of `Node` records, with a component `id` (the node's own identifier, e.g., its position in a list `nodes`) and a component `next` (a list of references to its children).

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* Users can define their own **data types**.
* A new **composite type definition** has the form
  ```julia
  struct <Name>
      <parts>
  end
  ```
* Each part, in addition to its **name**, can declare its **type** after `::`
* The name of the data type serves as **default constructor**, which when called creates a new object of this type, with given values assigned to each of the parts. 
</div>

In [ ]:
struct Node
    id
    next::Array{Node}
end

A single node with no children can then be made with the constructor that `julia` provides implicitly:

In [ ]:
Node(9, [])

The actual tree is created from its list of parents, where the root is its own parent:

In [ ]:
parents = [3,4,4,5,6,6,6]
nodes = [Node(i, []) for i in 1:7]
for (i,k) in pairs(parents)
    i == k || push!(nodes[k].next, nodes[i])
end
root = nodes[6]

A visitor function `pr` prints the id of each node it encounters:

In [ ]:
pr(x) = print(x.id, ", ")

With this visitor, BFS and DFS list all the nodes, in slightly different orders:

In [ ]:
BFS(root, pr);

In [ ]:
DFS(root, pr);

Compare the two orders with the tree:

![e7 tree](images/e7tree.png)

The recursive strategy can be adapted to a specific purpose, e.g., to print a tree as a tree:

In [ ]:
function tree_print(x, indent = "", first = true)
    first || print("\n", indent)
    print("-", x.id);
    first = true
    for c in x.next
        tree_print(c, indent * "  ", first)
        first = false
    end
end

In [ ]:
tree_print(root);

### Graph Traversal

* Both BFS and DFS can be applied to a (simple or directed) **graph**.
* Here, the **same node** then can possibly be reached through **different paths**.
* Thus, some care needs to be taken to **manage** these **repeat encounters**.
* The orbit algorithm below does this with a test `z in list`, before it adds a node $z$ to the list (see also Exercise 1.2).

###  Applications

* BFS visits the nodes in the order of their distance from the start, and so it finds shortest paths.  In an action graph, `orbit_with_words` finds a shortest word for each point (see *Words in the generators*), and `orbit_with_dist` (Exercise 1.5) records the distances.  Exercises 1.20 and 1.21 ask for shortest paths in a general graph.
* The nodes that can be reached from the start form its connected component.  In an action graph, this is the orbit.
* Part 2 finds the length of an element of a Coxeter group as such a distance, in the Cayley graph.  And Part 3 finds the connected components of a graph given by its edges, the classes of an equivalence relation, with Union-Find instead.

## Orbit Algorithms

<div class="alert alert-danger">

**Definition.** An **action** (from the right) of a group $G$ on a set $X$ is a map $X \times G \to X$ such that
1. $x.1 = x$ for all $x \in X$,
2. $x.(ab) = (x.a).b$ for all $x \in X$ and all $a, b \in G$.

</div>

* **Group actions** are a rich source of graphs.
* Here, the **nodes** of the graph are the elements $x$ of the domain $X$ that is acted upon.
* The **edges** of the graph are of the form $x \stackrel{s}{\longrightarrow} x.s$, implicitly given by the action.
* The elements $s$ typically come from a set of **generators** of the acting group.

<div class="alert alert-danger">

**Definition.**
Let $G$ be a group acting on a set $X$ via $(x, a) \mapsto x.a$, and suppose that $G = \langle S \rangle$ for 
some $S \subseteq G$. The **action graph** $\Gamma(S, X)$ is the (directed) graph with 
* **vertices:** $x \in X$, and
* **edges:** $x \stackrel{s}{\longrightarrow} x.s$ for $x \in X$, $s \in S$.
</div>

* The **orbit** of $x \in X$ under $G$ is the set $x^G = \{x.a : a \in G\}$.  It corresponds to the **connected component** of $x$ in the action graph $\Gamma(S, X)$.
* We formulate and apply a variant of BFS called **orbit algorithm** to compute orbits and other properties of actions of $G$.
* You'll be amazed by what all qualifies as an orbit.

### The Orbit Algorithm

To have some groups to play with, we import permutations from OrbitAl, together with a function `transpositions` that lists the transpositions of adjacent points:

In [ ]:
using OrbitAl: Perm, transpositions, cycles, shape

* For example, on $4$ points, the **simple transpositions** are $(1,2) = (^{1234}_{2134})$, $(2,3) = (^{1234}_{1324})$ and $(3,4) = (^{1234}_{1243})$. (Which group do they generate?)
* In the calculations below, we will use **one-line notation** $[2,1,3,4]$, $[1,3,2,4]$ and $[1,2,4,3]$ for these.

In [ ]:
n = 4
id = one(Perm, n)

In [ ]:
swaps = transpositions(n)

* Now suppose a finite group $G$ acts on a finite set $X$.
* For a given element $x \in X$, we wish to determine its orbit $x^G$ as the set of all the nodes in the graph $\Gamma(S, X)$ that can be reached from $x$ through the edges labelled by elements $s \in S$.
* This looks like a case for BFS with $y$.next ${} = \{y.s : s \in S\}$.

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* **Functions** are **first class objects**:
    * they can be passed as **arguments** into function calls and
    * they can be **returned as values** from function calls.

</div>

* The action of $G$ on $X$ is described by an **action function** `under`: calling `under(x, s)` returns $x.s$ (&ldquo;$x$ under $s$&rdquo;).
* For example, the **standard action** on points $(x, a) \mapsto x^a$ can be expressed as:

In [ ]:
onPoints(x, a) = x^a

Then $2^{(2,3)} = 3$:

In [ ]:
onPoints(2, swaps[2])

<div class="alert alert-info">
    
**Orbit Algorithm**
    
* **Input:** a list $A$ of **generating operators**, a **point** $x$ of the domain $X$, and an **action function** $(x, a) \mapsto x.a$. 
* **Output:** the **orbit** $x^G = \{x.g : g \in G\}$ of the point $x$ under the action of the group $G$ generated by $A$.
* **1.** **Initialize** a (growing) list $L$ to contain point $x$ only.
* **2.** **Loop** over each point $y$ in the growing list $L$, apply each generator $a$ in $A$ to $y$ and append any **new** result $z = y.a$ to $L$ (and stop when $L$ runs out of points).
* **3.** **Return** the resulting list $L$.
</div>

In pseudo-code:

![orbit](images/orbit.png)

* The following **implementation** serves as a **blueprint** for many variations that follow.
* Here, the **group action** function $(x, a) \mapsto x.a$ is provided as an argument `under`.  We will see the flexibility provided by this setup in the following examples.

In [ ]:
function orbit(aaa, x, under)
    list = [x]
    for y in list
        for a in aaa
            z = under(y, a)
            z in list || push!(list, z)
        end
    end
    return list
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* A tripled letter names a list of things named by that letter: `aaa` is a list of operators $a$, and later `xxx` is a list of points $x$, `eee` a list of elements (or of unit vectors $e_i$), and `www` a list of words.
* The logical operators `||` (OR) and `&&` (AND) are [short-circuiting](https://docs.julialang.org/en/v1/manual/control-flow/#Short-Circuit-Evaluation) and can be used to **abbreviate conditional statements**.

</div>

**Example.** To find the **orbit** of a point $x$ under the group generated by the swaps, apply the orbit algorithm to (i) the swaps, (ii) the point $x$, and (iii) the standard action **on points** $(x, a) \mapsto x^a$:

In [ ]:
orbit(swaps, 2, onPoints)

### Elements

The group $G$ acts on itself by **right multiplication**, $(x, a) \mapsto xa$, and $G$ is the orbit of any of its elements under this action, in particular of the identity.

In [ ]:
onRight(x, a) = x * a

So the orbit algorithm finds the **elements** of the group generated by the swaps, from the swaps, the identity permutation, and the action **on the right**:

In [ ]:
orbit(swaps, id, onRight)

In CGT, a group $G = \langle A \rangle$ is usually given by a **list $A$ of generators**, so that its elements need not be listed.  We follow this **design principle**: a new **composite type** `PermGp` holds a list `gens` of permutations, the **generators** of a **permutation group**, and its **identity element** `one`.  From now on, algorithms take a group as their input.

In [ ]:
struct PermGp
    gens::Array{Perm}
    one::Perm
end

In [ ]:
group = PermGp(swaps, id)
group.gens

The **elements of a group** are the orbit of its identity under its generators, sorted:

In [ ]:
elements(group::PermGp) = sort(orbit(group.gens, group.one, onRight))

For the group generated by the swaps, these are the $24$ permutations of $4$ points:

In [ ]:
eee = elements(group)

With the elements, we can **test membership** in a group, naively:

In [ ]:
import Base.in
in(a::Perm, group::PermGp) = a in elements(group)

In [ ]:
Perm([4,3,2,1]) in group

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* [in](https://docs.julialang.org/en/v1/base/collections/#Base.in)
* After `import Base.in`, the definition above adds a new **method** to Julia's own function `in`, for a `Perm` and a `PermGp`.
* Julia chooses among the methods of a function by the types of **all** its arguments ([multiple dispatch](https://docs.julialang.org/en/v1/manual/methods/)).

</div>

### Words in the generators

* Sometimes the question arises: how is a certain group element a **product of the generators**?
* In order to express an element of the group as **word** in the generators, we introduce an action on words.  
* For this, the generators $s \in S$ will be represented by symbols in $\mathbb{N} = \{1,2,3,\dotsc\}$ (Who is acting? On what?)
* The image of a word $w$ over $\mathbb{N}$ under an element $s$ of $\mathbb{N}$ is simply $ws$, the word obtained from $w$ by appending the letter $s$.

In [ ]:
onWords(word, s) = [word; s]

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* [vcat](https://docs.julialang.org/en/v1/base/arrays/#Base.vcat)

</div>

* The corresponding orbit algorithm is now required to produce **two lists in parallel**: the list `list` of elements as before, and a list `words` of corresponding words.
* We now need more control over the lists and make the indices `i` (in `list`) and `k` (in `aaa`) explicit.
* For this, we use `enumerate`, e.g., to loop over `list` with an explicit index `i`.

In [ ]:
function orbit_with_words(aaa, x, under)
    list = [x]
    words = [Int[]]
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            w = onWords(words[i], k)
            z in list || begin
                push!(list, z)
                push!(words, w)
            end
        end
    end
    return (list = list, words = words)
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* [named tuples](https://docs.julialang.org/en/v1/manual/functions/#Named-Tuples)
* [enumerate](https://docs.julialang.org/en/v1/base/iterators/#Base.Iterators.enumerate)

</div>

The new variant is used just like `orbit`.  Here it finds a word in the swaps for each element of the group they generate:

In [ ]:
www = orbit_with_words(swaps, id, onRight).words

* Note that, by construction, the orbit algorithm (being a form of BFS) finds a **word of shortest possible length** for each group element.
* In the above example, we can see how the group elements are enumerated by length, as a list that ends with a **unique word of longest length**.  Not every group might have such a unique longest element.

## Conjugacy Classes

* The group $G$ acts on itself by **conjugation**: $(x, a) \mapsto x^a = a^{-1} x a$.
* This action partitions the group $G$ into **conjugacy classes**.
* With suitable arguments, the action function `onPoints` from above also applies to the conjugation action of $G$ on itself.
* Hence, we can use the orbit algorithm to compute the conjugacy class of a single element in $G$.
* Perhaps surprisingly, it also finds the list of **all conjugacy classes** of $G$.

###  Conjugacy Class

To find the **conjugacy class** of an element $x$ of a group, apply the orbit algorithm to (i) the generators of the group, (ii) the element $x$, and (iii) the action `onPoints`, which here is conjugation $(x, a) \mapsto x^a = a^{-1} x a$:

In [ ]:
class(group::PermGp, x::Perm) = orbit(group.gens, x, onPoints)

The conjugacy class of a random element of the group:

In [ ]:
a = rand(eee)

In [ ]:
c = class(group, a)

In [ ]:
[cycles(a) for a in c]

In [ ]:
print([shape(a) for a in c])

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* [list comprehension](https://docs.julialang.org/en/v1/manual/arrays/#man-comprehensions)

</div>

* Sometimes it is convenient to treat the orbit $x^G$ of the point $x$ under the group $G$ as a pair $(G, x^G)$, where $x^G$ is a set, or a sorted list.
* Let's provide a data type `Orbit` for this.

In [ ]:
struct Orbit
    group
    elts  # sorted!
end

We denote (and construct) the class of `a` in `G` as `a^G`:

In [ ]:
import Base: ^
^(a::Perm, group::PermGp) = Orbit(group, sort(class(group, a)))

In [ ]:
cl = a^group

* We want to be able to test membership in, and to compare orbits: as sorted lists of elements, two classes are the same if their first elements are the same.
* Objects that are `==` must also have the same `hash`, since `Set` and `Dict` rely on it.  So `hash` gets a matching method: an orbit is hashed by its first element.

In [ ]:
import Base: in, isless, ==, hash
in(x, o::Orbit) = x in o.elts
==(o::Orbit, other::Orbit) = o.elts[1] == other.elts[1]
hash(o::Orbit, h::UInt) = hash(o.elts[1], h)
isless(o::Orbit, other::Orbit) = o.elts[1] < other.elts[1]

In [ ]:
cl == a^group

In [ ]:
cl == (a^0)^group

### All Conjugacy Classes

* To find all conjugacy classes of a group, the list of generators itself needs to be **closed under conjugation**.
* We need a version `orbitx` of the orbit algorithm that **initializes** its queue with **several points**, not just one.

In [ ]:
function orbitx(aaa, xxx, under)
    list = xxx
    for y in list
        for a in aaa
            z = under(y, a)
            z in list || push!(list, z)
        end
    end
    return list
end

For example, here are all conjugates of the swaps (what is this set usually called?):

In [ ]:
o = orbitx(swaps, copy(swaps), onPoints)

In [ ]:
[cycles(a) for a in o]

* Now, in order to find all conjugacy classes of the group $G$, we consider the following operation of $G$ on its conjugacy classes.
* The image of a conjugacy class $x^G$ under right multiplication with a group element $a$ is the conjugacy class $(xa)^G$.  (In what sense is this an action?  Is it even well-defined?  See Exercise 1.9.)

In [ ]:
onClasses(x, a) = (x.elts[1] * a)^(x.group)

To find all **conjugacy classes** of a group $G$, we close the set `gens` of generators of $G$ under conjugation, and compute the orbit of the class $1^G$ of the identity under the above action on classes:

In [ ]:
function conjClasses(gp::PermGp)
    orbit(orbitx(gp.gens, copy(gp.gens), onPoints), (gp.one)^gp, onClasses)
end

In [ ]:
cc = conjClasses(group)

Do the sizes of the conjugacy classes add up to the order of the group?  The size of an orbit is the length of its list of elements:

In [ ]:
import Base: size
size(o::Orbit) = length(o.elts)

In [ ]:
[size(c) for c in cc]

In [ ]:
n = 5
cc = conjClasses(PermGp(transpositions(n), Perm(n)))
sum(size.(cc))

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* A dot after a function name applies the function **elementwise**: `size.(cc)` is the list `[size(c) for c in cc]` ([dot syntax](https://docs.julialang.org/en/v1/manual/functions/#man-vectorized)).

</div>

In [ ]:
[shape(c.elts[1]) for c in cc]

<div class="alert alert-success">

**Remarks.**

* This method lists every element of every class, and so of the whole group: its time and memory grow with $|G|$.  But listing all the elements of a group is exactly what Computational Group Theory tries to avoid, wherever possible.  The Weyl group of type $E_7$, with $2\,903\,040$ elements, already needs about 10 GB of memory this way, even with the improvements of Exercise 1.28, and type $E_8$, with $696\,729\,600$ elements, is out of reach.
* The method is also wasteful: `onClasses` computes the whole class of $xa$ just to find out which class it is (see Exercise 1.28).
* The methods used in practice, in GAP [GAP] or Magma, represent a class by a single element.  They obtain the size of the class of $x$ as the index $|G : C_G(x)|$ of its centralizer, and test whether two elements are conjugate by a backtrack search, without listing any class [HEO05].
* For finite Coxeter groups, Part 2 uses their structure instead: each class has a canonical representative among its elements of minimal length [GP00].  This finds the $112$ classes of type $E_8$ without listing any of them.

</div>

## Subgroups

Next, we enumerate all subgroups of $G$ as an orbit.  For this, the operators need not form a group:

<div class="alert alert-success">

**Remarks**

* The operators `aaa` need not be invertible, nor do they need to generate a group: the orbit algorithm does not require the use of inverses.
* Neither do they need to generate a finite domain: the orbit algorithm can terminate if the list `aaa` and the orbit are finite.

</div>

* Sometimes, the operators `aaa` generate a **monoid**.  A **monoid action** (of a monoid $M$ on a set $X$) is defined in the same way as a group action (as a function $X \times M \to X$, $(x, m) \mapsto x.m$ satisfying the action axioms 1 and 2).
* A well-known monoid is the **power set** $2^S$ of a finite set $S$, with **set union** as its binary operation, generated by the **singleton sets** $\{s\}$, $s \in S$.

* Let $G$ be a group.
* The power set $(2^G, \cup)$ is a monoid, generated by the singletons $\{a\}$, $a \in G$, as atoms.
* $2^G$ acts on the subgroups $H$ of $G$ via **closure**: $H.A = \langle H, A \rangle$.  (Check!)

In [ ]:
closure(group::PermGp, a::Perm) = PermGp(union(group.gens, [a]), group.one)

In [ ]:
onGroups(x, a) = closure(x, a)

The same subgroup arises from different lists of generators, and the orbit algorithm must recognize a subgroup it has seen before.  So two groups are compared by their elements, with a matching `hash`, and `isless` sorts them, as for the `Orbit` objects above:

In [ ]:
==(group::PermGp, other::PermGp) = elements(group) == elements(other)
hash(group::PermGp, h::UInt) = hash(elements(group), h)
isless(group::PermGp, other::PermGp) = elements(group) < elements(other)

In fact, each subgroup of $G$ lies in the $2^G$-orbit of the trivial subgroup (check!), and so the orbit algorithm finds all subgroups:

In [ ]:
subgroups(group) = orbit(elements(group), PermGp([], group.one), onGroups)

Let's apply this to our group, generated by the swaps.  Its size is the number of its elements:

In [ ]:
size(group::PermGp) = length(elements(group))

In [ ]:
size(group)

In [ ]:
subs = subgroups(group)

In [ ]:
length(subs)

In [ ]:
print(size.(subs))

### Conjugacy Classes of Subgroups

* The group $G$ acts on its set of subgroups by conjugation: $(H, a) \mapsto H^a = \{h^a : h \in H\}$.
* Let's implement this as a julia operator `^`, together with a function `subgpClass` that computes the conjugacy class $H^G$ of a given subgroup $H$ of $G$ as $G$-orbit under this action.
* We also define `H^G` as an `Orbit` object.

In [ ]:
^(group::PermGp, a::Perm) = PermGp([x^a for x in group.gens], group.one)
subgpClass(gp::PermGp, subgp::PermGp) = orbit(gp.gens, subgp, onPoints)
^(subgp::PermGp, group::PermGp) = Orbit(group, sort(subgpClass(group, subgp)))

Combining ideas from above, we now define the image of a conjugacy class of subgroups $H^G$ under a singleton $\{a\}$ as the conjugacy class $\langle H, a \rangle^G$.  (Is this a well-defined action?  See Exercise 1.14.)

In [ ]:
onSubgpClasses(x, a) = onGroups(x.elts[1], a)^(x.group)

To compute all **conjugacy classes of subgroups** of $G$, we determine the orbit of the class of the trivial subgroup under the monoid $(2^G, \cup)$ with respect to that action:

In [ ]:
function subgpClasses(gp::PermGp)
    orbit(elements(gp), PermGp([], gp.one)^gp, onSubgpClasses)
end

In [ ]:
ccs = subgpClasses(group);

In [ ]:
length(ccs)

In [ ]:
print(size.(ccs))

In [ ]:
print([size(c.elts[1]) for c in ccs])

In [ ]:
sum(size.(ccs))

For the symmetric group on $5$ points, there are $19$ classes of subgroups, with $156$ subgroups in all:

In [ ]:
ccs = subgpClasses(PermGp(transpositions(5), one(Perm, 5)));
length(ccs)

In [ ]:
sum(size.(ccs))

<div class="alert alert-success">

**Remarks.**

* This naive application of the orbit algorithm is obviously not suitable for larger examples.  But it can still serve as high level description for more refined implementations.
* For more counting results on the subgroups of the symmetric group see sequences [A000638](https://oeis.org/A000638) and [A005432](https://oeis.org/A005432) of the **Online Encyclopedia of Integer Sequences**.

</div>

## Stabilizer and Transversal

The performance of certain orbit calculations can be drastically improved by making systematic use of stabilizer subgroups.

<div class="alert alert-danger">

**Definition.**  Let $G$ be a finite group acting on a set $X$.  The **stabilizer** of the point $x \in X$ is the set $G_x = \{a \in G : x.a = x\}$.  Clearly, $G_x$ is a subgroup of $G$.

</div>

* We can use a variant of the orbit algorithm to determine (and remember), for each point $y$ in the $G$-orbit of $x$, a representative element $t_y \in G$ with $x.t_y = y$.
* This list of representatives will form a **transversal** of the cosets of the stabilizer of $x$ in $G$.
* By Schreier's Lemma, a generating set for the **stabilizer** can be computed from the transversal.

### Transversal

* We initialize an additional list `reps` with the identity element, mapping $x$ to itself: $x.1 = x$
* Whenever a new point $z \in X$ is found, as image under $a \in A$ of a point $y \in X$, we add $t_z := t_y a \in G$ to the list `reps`.

In [ ]:
function orbit_with_transversal(aaa, x, under)
    list = [x]
    reps = [aaa[1]^0]
    for (i, y) in enumerate(list)
        for a in aaa
            z = under(y, a)
            t = reps[i] * a
            z in list || begin
                push!(list, z)
                push!(reps, t)
            end
        end
    end
    return (list = list, reps = reps)
end

In [ ]:
swaps = transpositions(5)
transversal = orbit_with_transversal(swaps, 2, onPoints)

### Stabilizer

<div class="alert alert-danger">

**Schreier's Lemma.**
Suppose a group $G$, generated by a set $A$,
acts on a set $X$ and that $\{t_y : y \in x^G\}$ is a transversal
of the orbit of $x \in X$.  Then the set
$$
    \{t_y a t_{y.a}^{-1}: a \in A,\, y \in x^G\}
$$
generates the stabilizer $G_x$ of $x$ in $G$.
    
</div>

* In the next variant of the orbit algorithm, we also collect the Schreier generators $t_ya t_{y.a}^{-1}$ in a list `stab`.
* For each newly constructed point $z := y.a$, we need to determine its **exact position** in `list` if it has been seen before.
* For this, we now maintain an `index` dictionary, which assigns to each point its index in `list`.  The function `get!` will find this index, or update `index` if point `z` is indeed new.

In [ ]:
function orbit_with_stabilizer(aaa, x, under)
    list = [x]
    index = Dict(x => 1)
    reps = [aaa[1]^0]
    stab = Perm[]
    for (i, y) in enumerate(list)
        for a in aaa
            z = under(y, a)
            t = onRight(reps[i], a)
            l = get!(index, z) do # index of z in list
                push!(list, z)
                push!(reps, t)
                length(list)
            end    # x^(reps[i] * a) = x^reps[l]
            t == reps[l] || push!(stab, t / reps[l])
        end
    end
    return (list = list, reps = reps, stab = stab)
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* A **dictionary** `Dict(x => 1)` stores **values** under **keys**.  Here, `index` stores the position of each point in `list`, so that it can be found without searching through `list` ([dictionaries](https://docs.julialang.org/en/v1/base/collections/#Dictionaries)).
* `get!(f, dict, key)` returns `dict[key]` if it exists; otherwise it stores the value `f()` under `key` and returns that.
* A **do-block** `get!(dict, key) do ... end` is another way to write `get!(() -> ..., dict, key)`: the block becomes a function without a name, passed as the first argument ([do-block syntax](https://docs.julialang.org/en/v1/manual/functions/#Do-Block-Syntax-for-Function-Arguments)).

</div>

In [ ]:
stabilizer = orbit_with_stabilizer(swaps, 2, onPoints)

In [ ]:
size(PermGp(stabilizer.stab, swaps[1]^0))

In [ ]:
length(stabilizer.stab)

In [ ]:
setdiff(stabilizer.stab, [Perm(5)])

<div class="alert alert-danger">

**Orbit-Stabilizer Lemma.**  Let $G$ act on $X$, and let $x \in X$.  Then $G_x a \mapsto x.a$ is a bijection from the right cosets of $G_x$ in $G$ to the orbit $x^G$.  In particular, $|x^G| = |G : G_x|$, so $|G| = |x^G| \, |G_x|$, and the transversal $\{t_y : y \in x^G\}$ is a set of coset representatives.

</div>


Here, $5$ points times a stabilizer of order $24$ gives the order $120$ of the symmetric group:

In [ ]:
length(stabilizer.list) * size(PermGp(stabilizer.stab, swaps[1]^0)) == size(PermGp(swaps, swaps[1]^0))


Counting the pairs $(x, a)$ with $x.a = x$ in two ways, once by $a$ and once by $x$, and using the Orbit-Stabilizer Lemma on each orbit, gives a formula for the number of orbits.

<div class="alert alert-danger">

**Cauchy-Frobenius Lemma.**  Let a finite group $G$ act on a finite set $X$.  Then the number of $G$-orbits on $X$ is the average number of fixed points,
$$
  \frac{1}{|G|} \sum_{a \in G} |\{x \in X : x.a = x\}|.
$$

</div>


* A **necklace** of length $n$ with $k$ colours is an orbit of the cyclic group of rotations on the words of length $n$ in $k$ letters.
* Here are the binary necklaces of length $6$, as orbits under the action that rearranges the letters of a word (see Exercise 1.7):


In [ ]:
using OrbitAl: permuted
C6 = PermGp([Perm([2,3,4,5,6,1])], Perm(6))
binary = [digits(i, base = 2, pad = 6) for i in 0:63]
necklaces = Set(Set(orbit(C6.gens, w, permuted)) for w in binary)
length(necklaces)


Their number, by the Cauchy-Frobenius Lemma, is the average number of fixed points:


In [ ]:
fixed(a) = count(w -> permuted(w, a) == w, binary)
sum(fixed(a) for a in elements(C6)) // size(C6)


The rotation by $i$ places has $\gcd(i, n)$ cycles, and a word is fixed by it if and only if it is constant on each cycle.  So it fixes $k^{\gcd(i, n)}$ words, and no orbit needs to be computed at all:

In [ ]:
sum(2^gcd(i, 6) for i in 0:5) // 6


## Stabilizer Chain

In [ ]:
import OrbitAl: isidentity, last_moved
is_trivial(group::PermGp) = all(isidentity, group.gens)
last_moved(group::PermGp) = max(last_moved.(group.gens)...)

* Schreier's Lemma gives generators of the stabilizer, which can be subjected to further orbit calculations.
* Applying the above ideas along a chain of stabilizers can yield information about the whole group.

### Order of the Group

By the Orbit-Stabilizer Lemma, along the stabilizer chain, $|G| = |x^G| \, |G_x| = |x^G| \, |y^{G_x}| \, |G_{x, y}| = {\dots}$.  So the order of $G$ is the product of the orbit lengths along the chain:

In [ ]:
function sizeOfGroup(group)
    is_trivial(group) && return 1
    x = last_moved(group)
    orb = orbit_with_stabilizer(group.gens, x, onPoints)
    stab = PermGp(setdiff(orb.stab, [group.one]), group.one)
    return sizeOfGroup(stab) * length(orb.reps)
end

In [ ]:
sizeOfGroup(group)

Some slightly bigger examples are the symmetric group on $10$ points, and the largest **Mathieu group** $M_{24}$:

In [ ]:
group = PermGp(transpositions(10), Perm(10))

In [ ]:
sizeOfGroup(group)

In [ ]:
m24 = PermGp([
  Perm([2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,1,24]),
  Perm([1,2,17,13,4,6,9,18,3,7,12,23,14,19,20,15,10,11,5,22,16,21,8,24]),
  Perm([24,23,12,16,18,10,20,14,21,6,17,3,22,8,19,4,11,5,15,7,9,13,2,1]),
], Perm(24))
sizeOfGroup(m24)

###  Random Element

Random elements of a group are found along the same lines as `sizeOfGroup`, with a random coset representative at each link of the chain:

In [ ]:
function randomGroupElement(group)
    is_trivial(group) && return group.one
    x = last_moved(group)
    orb = orbit_with_stabilizer(group.gens, x, onPoints)
    stab = PermGp(setdiff(orb.stab, [group.one]), group.one)
    return randomGroupElement(stab) * rand(orb.reps)
end

In [ ]:
a = randomGroupElement(m24)

These random elements are **uniformly distributed**: in $2400$ trials, each of the $24$ elements of the symmetric group on $4$ points turns up about $100$ times:

In [ ]:
group = PermGp(transpositions(4), Perm(4))
count = Dict(a => 0 for a in elements(group))
for i in 1:2400
    count[randomGroupElement(group)] += 1
end
count

### Membership

Is $a \in G$?  This is decided along the same lines: provided that $y = x^a$ lies in the $G$-orbit of $x$, we have $a \in G \iff a/t_y \in G_x$.  Otherwise, $a \notin G$.

In [ ]:
function memberOfGroup(group::PermGp, perm::Perm)
    is_trivial(group) && return perm == group.one
    x = last_moved(group)
    orb = orbit_with_stabilizer(group.gens, x, onPoints)
    l = findfirst(==(x^perm), orb.list)
    isnothing(l) && return false
    stab = PermGp(setdiff(orb.stab, [group.one]), group.one)
    return memberOfGroup(stab, perm / orb.reps[l])
end

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">

* `findfirst(f, list)` is the position of the first entry `x` of `list` with `f(x) == true`, or `nothing` if there is none.  `isnothing(l)` tests for that.
* `==(y)` is the function `x -> x == y`.

</div>

In [ ]:
memberOfGroup(m24, a)

In [ ]:
b = rand(Perm, 24)

In [ ]:
memberOfGroup(m24, b)

Similar ideas express an element as a **word in the generators**, $a = s_1 \dotsm s_k$, and so evaluate **homomorphisms** defined on generators: $\phi(a) = \phi(s_1) \dotsm \phi(s_k)$.

* The number of Schreier generators grows fast along the chain.  For the **Rubik's cube group** [Joy08], on $48$ points, `sizeOfGroup` meets $6$, $94$, $2207$, $46264$ and $1016766$ generators for the first five groups in the chain, and does not finish.
* The **Schreier-Sims** algorithm [Sim70] keeps only the Schreier generators that are really needed.  Part 5 develops it.  OrbitAl's `SimsGp` uses it, and finds the order of the cube group at once:

In [ ]:
using OrbitAl.permgroups: cube
using OrbitAl.simsgroup: SimsGp
size(SimsGp(cube, Perm(48)))

## Graphs

* With a further small modification, the orbit algorithm can keep track of edges and thus construct the **action graph**.  
* Recall that this is a labelled directed graph, with vertices $x, y \in X$ and edges $x \stackrel{s}{\to} y$, whenever $x.s = y$ for $x, y \in X$, and $s \in S$.  
* We need to decide on a **data structure** for such graphs.  The simplest, perhaps, is a **list of pairs** of indices, each representing an edge.

In [ ]:
using Graphs
using GraphPlot
import Cairo, Fontconfig

In [ ]:
elist = [(1,2),(1,3),(1,4),(2,2)]
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

The tree from the beginning becomes a simple graph, with an edge joining each node $i$ to its parent:

In [ ]:
collect(enumerate(parents))

In [ ]:
graph = SimpleGraph(Edge.(enumerate(parents)))
gplot(graph, nodelabel=vertices(graph))

The next variant of the orbit algorithm keeps track of the edges in a set `edges`, as pairs of positions in `list`, leaving out loops:

In [ ]:
function orbit_with_edges(aaa, x, under)
    list = [x]
    index = Dict(x => 1)
    edges = Set{Tuple{Int, Int}}()
    for (i, y) in enumerate(list)
        for a in aaa
            z = under(y, a)
            l = get!(index, z) do
                push!(list, z)
                length(list)
            end
            i == l || push!(edges, (i, l))
        end
    end
    return (list = list, edges = collect(edges))
end

The edges of the action of the swaps on the points, and the graph they form, a path:

In [ ]:
elist = sort(orbit_with_edges(swaps, 1, onPoints).edges);
print(elist)

In [ ]:
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

* The action of $G$ on $X$ induces an action on the pairs $(x, y) \in X \times X$: $((x, y), a) \mapsto (x.a, y.a)$.
* We can define a corresponding action function `onPairs`.

<div class="alert alert-warning">

<img src="images/julia.png" alt="julia" width="30">  

* [Pair](https://docs.julialang.org/en/v1/base/collections/#Core.Pair)
* [Set](https://docs.julialang.org/en/v1/base/collections/#Base.Set)
</div>

In [ ]:
onPairs(pair::Pair, a::Perm) = pair[1]^a => pair[2]^a

In [ ]:
elist = orbit_with_edges(swaps, 1 => 2, onPairs).edges;
print(elist)

In [ ]:
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

* The action of $G$ on $X$ induces an **action on the subsets** $Y \in 2^X$: $(Y, a) \mapsto Y^a = \{y^a : y \in Y\}$.
* We can implement a corresponding action function `onSets`.

In [ ]:
onSets(set::Set, a ::Perm) = Set([x^a for x in set])

In [ ]:
elist = orbit_with_edges(transpositions(6), Set([1,2]), onSets).edges;
print(elist)

In [ ]:
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

Finally, the swaps acting by right multiplication on the group they generate give the **Cayley graph** of all permutations of $4$ points:

In [ ]:
elist = orbit_with_edges(transpositions(4), Perm(4), onRight).edges;
print(elist)

In [ ]:
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

### Permutations

* The action graph is in fact a directed graph.
* It encodes the permutations induced by the action on the domain.
* For this (and other applications) it will be more convenient to store the edge information in a different format: as lists of `images`.

In [ ]:
function orbit_with_images(aaa, x, under)
    list = [x]
    index = Dict(x => 1)
    images = [Int[] for a in aaa]
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = under(y, a)
            l = get!(index, z) do
                push!(list, z)
                length(list)
            end
            push!(images[k], l)
        end
    end
    return (list = list, images = images)
end

The swaps act on the points $1, \dots, 5$ by these image lists:

In [ ]:
swaps = transpositions(5)
orb = orbit_with_images(transpositions(5), 1, onPoints)

Each image list is a permutation in one-line notation:

In [ ]:
Perm.(orb.images)

The swaps also act on the $10$ sets of two points, as permutations of degree $10$:

In [ ]:
orb = orbit_with_images(swaps, Set([1,2]), onSets)
perms = Perm.(orb.images)

In [ ]:
for perm in perms
    println(cycles(perm))
end

When the images are converted into edges, it helps to be able to list entries together with their position in a list.  This can be regarded as the transpose of the usual two-line notation for permutations:

In [ ]:
collect(enumerate(orb.images[1]))

In [ ]:
elist = union(enumerate.(orb.images)...)
elist = [(x, y) for (x, y) in elist if x != y]
graph = SimpleGraph(Edge.(elist))
gplot(graph, nodelabel=vertices(graph))

## Exercises

**Exercise 1.1.**  Is Euclid's Algorithm BFS or DFS?  What is the tree behind it, vertices, edges?  Make the tree explicit and use it to formulate the [Extended Euclidean Algorithm](https://en.wikipedia.org/wiki/Extended_Euclidean_algorithm), which expresses the gcd $d$ of $a$ and $b$ as a linear combination $d = ax + by$.

<div class="alert alert-success">
<details>
<summary><b>Answer.</b></summary>

The recursion visits the pairs $(a, b) \to (b, a \bmod b) \to \dots \to (d, 0)$: these are the vertices of the tree, with an edge from each pair to the next.  So the tree is a path, and on a path, BFS and DFS visit the vertices in the same order.  But the extended algorithm also needs a second visit to each vertex, on the way back, after its child is done, and this is DFS.

On the way down, the tree is built.  On the way back up, the coefficients are passed from each child to its parent: with $q = a \div b$, the child $(b, a \bmod b)$ has $d = b x + (a \bmod b) y$, and $a \bmod b = a - q b$, so $d = a y + b (x - q y)$.  At the leaf $(d, 0)$, $d = d \cdot 1 + 0 \cdot 0$.

</details>
</div>

In [ ]:
# recursive, like gcd: if d = b x + (a mod b) y, then d = a y + b (x - (a ÷ b) y)
function gcdex(a, b)
    b == 0 && return (1, 0)
    q, r = divrem(a, b)
    x, y = gcdex(b, r)
    return (y, x - q * y)
end

# the tree made explicit: a path from (a, b) down to (d, 0)
mutable struct GcdNode
    a
    b
    xy                      # eventually gcd(a, b) = a x + b y
    next::Vector{GcdNode}
end
GcdNode(a, b) = GcdNode(a, b, (1, 0), GcdNode[])

# DFS with two visitors: one on the way down, one on the way back up
function DFS2(x, visit1, visit2)
    visit1(x)
    for z in x.next
        DFS2(z, visit1, visit2)
        visit2(x)
    end
end

function gcdex_tree(a, b)
    # extend the tree: the child of (a, b) is (b, a mod b)
    rise(node) = node.b == 0 || push!(node.next, GcdNode(node.b, node.a % node.b))
    # pass the coefficients back up
    function fall(node)
        x, y = node.next[1].xy
        node.xy = (y, x - div(node.a, node.b) * y)
    end
    root = GcdNode(a, b)
    DFS2(root, rise, fall)
    return root.xy
end

a, b = 240, 46
x, y = gcdex(a, b)
(x, y), a * x + b * y == gcd(a, b), gcdex_tree(a, b) == (x, y)

**Exercise 1.2** (Graph Traversal).  Modify BFS and DFS so that it can be used to visit the nodes of a (connected, simple) graph, taking care of repeat encounters.

**Exercise 1.3.**  Prove the correctness of the Orbit Algorithm: Show that it terminates and that after termination $L = \{x.g : g \in G\}$, where $G = \langle A \rangle$.

<div class="alert alert-success">
<details>
<summary><b>Proof.</b></summary>

Let $L$ be the final list and $x^G = \{x.g : g \in G\}$ the orbit of $x$ under $G = \langle A \rangle$.

**Termination.** Every element added to $L$ belongs to $x^G$ (shown below).  Since $G$ is finite, $x^G$ is finite, and the `z in list` guard ensures each element is pushed at most once, so the loop terminates.

**$L \subseteq x^G$.**  Induction on when elements are added:
* Initially $x = x.1 \in x^G$.
* If $y \in L$ with $y = x.g$, and $a \in A$, then $z = y.a = x.(ga) \in x^G$ since $ga \in G$.

**$x^G \subseteq L$.**  Since $G$ is finite, every $g \in G = \langle A \rangle$ can be written as a product of elements of $A$ without inverses: for any $a \in A$, $a^{-1} = a^{\mathrm{ord}(a)-1}$ is a positive power of $a$.  So let $y = x.(a_1 \cdots a_k)$ with each $a_i \in A$.  Induction on $k$:
* $k = 0$: $y = x \in L$. $\checkmark$
* $k > 0$: Set $y' = x.(a_1 \cdots a_{k-1})$.  By hypothesis $y' \in L$, so $y'$ is eventually processed in the outer loop.  At that point $z = y'.a_k = y$ is computed and added to $L$ if not already present.  Hence $y \in L$. $\checkmark$

The last step relies on the `for y in list` loop ranging over the dynamically growing list, so every element that enters $L$ is eventually processed. $\blacksquare$

</details>
</div>

**Exercise 1.4.**  Draw the action graph of $M_{24}$ on the $24$ points.

**Exercise 1.5** (Orbit with Distance).  Write a version `orbit_with_dist` of the orbit algorithm that computes and returns the **shortest distance** 
  from the starting point of each point in the orbit, in addition to the list of points. Apply this algorithm to the action of $M_{24}$ on the $24$ points with starting point $1$.

In [ ]:
function orbit_with_dist(aaa, x, under)
    # your code here: return (list = ..., dist = ...)
end

In [ ]:
import OrbitAl
m24 = PermGp([
  Perm([2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,1,24]),
  Perm([1,2,17,13,4,6,9,18,3,7,12,23,14,19,20,15,10,11,5,22,16,21,8,24]),
  Perm([24,23,12,16,18,10,20,14,21,6,17,3,22,8,19,4,11,5,15,7,9,13,2,1]),
], Perm(24))
mine = orbit_with_dist(m24.gens, 1, onPoints)
theirs = OrbitAl.orbit_with_dist(m24.gens, 1, onPoints)
!isnothing(mine) && Dict(zip(mine.list, mine.dist)) == Dict(zip(theirs.list, theirs.dist))

**Exercise 1.6.**  Write a version of the orbit algorithm that takes **multiple starting points** and computes for each newly found point, in parallel to the list of points, the **shortest distance** to any of the starting points.
  Apply this algorithm to the action of $M_{24}$ on $24$ points, with starting points $1, 2, 3$.

In [ ]:
function orbitx_with_dist(aaa, xxx, under)
    # your code here: return (list = ..., dist = ...)
end

In [ ]:
import OrbitAl
mine = orbitx_with_dist(m24.gens, [1, 2, 3], onPoints)
theirs = OrbitAl.orbitx_with_dist(m24.gens, [1, 2, 3], onPoints)
!isnothing(mine) && Dict(zip(mine.list, mine.dist)) == Dict(zip(theirs.list, theirs.dist))

**Exercise 1.7** (Pólya Action).  The symmetric group of degree $n$ acts on the words of length $n$ (over a finite alphabet $A$) by rearranging the letters of a given word.  In `OrbitAl`, this action is called `permuted`.  Compute the orbit of the word `"11222"` under the action, find the permutations induced by the action on the list of all words in the orbit, plot the action graph and determine the stabilizer of the word.

In [ ]:
using OrbitAl: permuted
word = [1,1,2,2,2]
swaps = transpositions(length(word))
orb = orbit_with_images(swaps, word, permuted)
stab = orbit_with_stabilizer(swaps, word, permuted)
orb.list

**Exercise 1.8** (Cosets).  Let $H$ be a subgroup of a group $G$. Show that $G$ acts on the (right) cosets $\{Ha : a \in G\}$ of $H$ in $G$ via $Ha.b = H.ab$.  Implement a function `cosets(G, H)` that computes the list of all (right) cosets of $H$ in $G$ as an application of the orbit algorithm.  Then determine the cosets of $\langle (1,2), (3,4) \rangle$ in the symmetric group on $4$ points.

In [ ]:
function cosets(G::PermGp, H::PermGp)
    # your code here: return the list of all right cosets of H in G, each as a list of its elements
end

In [ ]:
import OrbitAl
G = PermGp(transpositions(4), Perm(4))
H = PermGp([Perm([2,1,3,4]), Perm([1,2,4,3])], Perm(4))
mine = cosets(G, H)
theirs = OrbitAl.coset.cosets(OrbitAl.PermGp(G.gens, G.one), OrbitAl.PermGp(H.gens, H.one))
elts(c) = Set(h * c.element for h in OrbitAl.elements(c.subgp))
!isnothing(mine) && Set(Set.(mine)) == Set(elts.(theirs))

**Exercise 1.9** (Conjugacy Classes).  Show that the partition of a (finite) group into conjugacy classes can be computed as an orbit under right multiplication, provided that the set of generators is closed under conjugation.

<div class="alert alert-success">
<details>
<summary><b>Proof.</b></summary>

**Not an action.**  The class $(xa)^G$ depends on the choice of $x$ in its class: in $S_3$, the transpositions $x = (1,2)$ and $x' = (1,3)$ are conjugate, but $x \cdot (1,2) = 1$, while $x' \cdot (1,2)$ is a $3$-cycle.  So $x^G \mapsto (xa)^G$ is not even a map on the classes.  But `onClasses` uses a fixed representative of each class, its first element, and so it is a map on the classes that the orbit algorithm meets.  And the orbit algorithm needs no more than that: the operators need not form a group, nor even act (see the Remarks under *Subgroups*).

**Only classes.**  Each point that the orbit algorithm adds to the list is a conjugacy class of $G$, by construction.

**All classes.**  Let $S$ be the generators, closed under conjugation.  Since $G$ is finite, each $g \in G$ is a product $g = s_1 \cdots s_k$ of elements of $S$, without inverses.  Induction on $k$ shows that the class of $g$ is in the list:
* $k = 0$: the list starts with the class $1^G$.
* $k > 0$: by induction, the class of $g' = s_1 \cdots s_{k-1}$ is in the list, with some representative $r = g'^h$, $h \in G$.  The orbit algorithm applies each $s \in S$ to it, in particular $s = s_k^h$, which lies in $S$, since $S$ is closed under conjugation.  This gives the class of $r s_k^h = (g' s_k)^h = g^h$, which is the class of $g$.  $\blacksquare$

</details>
</div>

**Exercise 1.10** (Subgroups).  For a finite group $G$, show that the monoid $(2^G, \cup)$ acts on the set of subgroups $H$ of $G$ via **closure**: $H.A = \langle H, A\rangle$, and that each subgroup of $G$ lies in the orbit of the trivial subgroup under this action.

**Exercise 1.11.**  Show that $(a, n) \mapsto a^n$ does not define an action of the (additive) group of the integers $\mathbb{Z}$ on a finite group $G$. However, for any $x \in G$, the map $(a, n) \mapsto a x^n$ does define an action of $\mathbb{Z}$ on $G$.

**Exercise 1.12.**  The **zuppos** of a finite group $G$ are the cyclic subgroups of prime power order (**z**yklische **U**ntergruppen von **P**rimzahl**p**otenz**o**rdnung).  Let $Z \subseteq G$ be a set that contains exactly one generator of each zuppo.  Describe and implement an algorithm that constructs $Z$.  Show that the singletons $\{z\}$, $z \in Z$, suffice as generators to find all subgroups of $G$ under the closure action of $2^G$.

In [ ]:
function zuppos(group::PermGp)
    # your code here: return a list containing exactly one generator of each zuppo
end

In [ ]:
import OrbitAl
G = PermGp(transpositions(5), Perm(5))
mine = zuppos(G)
theirs = OrbitAl.zuppos(OrbitAl.PermGp(G.gens, G.one))
cyclic(z) = Set(z^k for k in 1:OrbitAl.order(z))
!isnothing(mine) && Set(cyclic.(mine)) == Set(cyclic.(theirs))

**Exercise 1.13** (Cyclic Extension).  Let $G$ be a finite **solvable** group, so that each subgroup $U \neq 1$ of $G$ has a normal subgroup $V$ of prime index $p$ (why?).  Show that then $U = \langle V, z \rangle$ for a generator $z$ of a zuppo (Exercise 1.12) with $z \notin V$, $V^z = V$ and $z^p \in V$.  So all subgroups of $G$ can be found from the trivial subgroup by such **cyclic extensions** [Neu60], a restricted version of the closure action of Exercise 1.10.  Write a function `cyclicExtensions(group)` that returns the list of all subgroups of a solvable group in this way.  Compare the number of closures it computes with that of `subgroups`, for $S_4$ and for the wreath product $S_3 \wr S_2$, of order $72$.  What happens for $S_5$, which is not solvable?


In [ ]:
function cyclicExtensions(group::PermGp)
    # your code here: return the list of all subgroups of the solvable group, by cyclic extension from the trivial subgroup
end


In [ ]:
function check_cyclic()
    S4 = PermGp(transpositions(4), Perm(4))
    mine = cyclicExtensions(S4)
    !isnothing(mine) && Set(mine) == Set(subgroups(S4))
end
check_cyclic()


<div class="alert alert-success">
<details>
<summary><b>Answer.</b></summary>

**Why $V$ exists.**  The subgroup $U \neq 1$ is solvable, so its derived subgroup $U'$ is a proper subgroup, and the abelian group $U/U'$ has a subgroup of prime index $p$.  Its preimage $V$ is normal in $U$, of index $p$.

**Why it works.**  As $p$ divides $|U|$ more often than $|V|$, a Sylow $p$-subgroup of $U$ is not contained in $V$, and so contains an element $x \notin V$.  Its cyclic subgroup $\langle x \rangle$ is a zuppo, with generator $z$, say, and $z \notin V$, as $\langle z \rangle = \langle x \rangle$.  Then $z \in U$ normalizes $V$, and $z^p \in V$, as $U/V$ has order $p$.  And $V < \langle V, z \rangle \leq U$, where $|U : V| = p$ is a prime, so $U = \langle V, z \rangle$.  As $V$ is again solvable, induction on $|U|$ shows that each subgroup of $G$ is found from the trivial subgroup.

**Fewer closures.**  `subgroups` applies all elements of $G$ to all subgroups: $24 \cdot 30 = 720$ closures for $S_4$, and $72 \cdot 112 = 8064$ for $S_3 \wr S_2$.  Cyclic extension applies only the $16$ and $34$ zuppo generators, and computes a closure only when the test is passed: $95$ and $628$ closures.  For $S_3 \wr S_2$, this brings the time down by a factor of about $7$.

**$S_5$.**  Cyclic extension finds only $154$ of the $156$ subgroups of $S_5$.  The alternating group $A_5$ is perfect, $A_5' = A_5$, so it has no normal subgroup of prime index, and it is never found.  Neither is $S_5$, whose only normal subgroup of prime index is $A_5$.  In general, the perfect subgroups have to be supplied as additional starting points.

</details>
</div>


In [ ]:
import OrbitAl
# a generator of each zuppo, from OrbitAl (Exercise 1.12)
zupposOf(group::PermGp) = OrbitAl.zuppos(OrbitAl.PermGp(group.gens, group.one))

# V.z = <V, z>, if z is not in V, normalizes V, and z^p is in V, for the prime p dividing the order of z; and V.z = V otherwise
function cyclicExtensions(group::PermGp)
    function extend(V, z)
        elts = elements(V)
        n = OrbitAl.order(z)
        p = first(p for p in 2:n if n % p == 0)
        z in elts && return V
        z^p in elts || return V
        all(v^z in elts for v in V.gens) || return V
        return closure(V, z)
    end
    return orbit(zupposOf(group), PermGp([], group.one), extend)
end

S3wrS2 = PermGp([Perm([2,1,3,4,5,6]), Perm([2,3,1,4,5,6]), Perm([4,5,6,1,2,3])], Perm(6))
t_subgroups = @elapsed all_subs = subgroups(S3wrS2)
t_cyclic = @elapsed cyc_subs = cyclicExtensions(S3wrS2)
subs5 = cyclicExtensions(PermGp(transpositions(5), Perm(5)))     # no subgroup of order 60 or 120
check_cyclic(), Set(cyc_subs) == Set(all_subs), round(t_subgroups / t_cyclic, digits = 1), length(subs5), maximum(size.(subs5))


**Exercise 1.14.**  Show that the conjugacy classes of subgroups of a finite group can be computed as an orbit under the action of its power set.

<div class="alert alert-success">
<details>
<summary><b>Proof.</b></summary>

As in Exercise 1.9, $H^G \mapsto \langle H, a \rangle^G$ is not a map on the classes: in $S_3$, the subgroups $H = \langle (1,2) \rangle$ and $H' = \langle (1,3) \rangle$ are conjugate, but $\langle H, (1,2) \rangle = H$ has order $2$, while $\langle H', (1,2) \rangle = S_3$.  Again, `onSubgpClasses` uses a fixed representative of each class, and each point added to the list is a conjugacy class of subgroups.

**All classes.**  Each subgroup $H$ of $G$ is generated by finitely many elements, $H = \langle a_1, \dots, a_k \rangle$.  Induction on $k$ shows that the class of $H$ is in the list:
* $k = 0$: the list starts with the class of the trivial subgroup.
* $k > 0$: by induction, the class of $H' = \langle a_1, \dots, a_{k-1} \rangle$ is in the list, with some representative $R = H'^h$, $h \in G$.  The orbit algorithm applies each $a \in G$ to it, in particular $a = a_k^h$.  This gives the class of $\langle R, a_k^h \rangle = \langle H', a_k \rangle^h = H^h$, which is the class of $H$.  $\blacksquare$

Here, all elements of $G$ serve as operators, a set that is closed under conjugation, as the generators in Exercise 1.9 must be.

</details>
</div>

**Exercise 1.15.**  Prove Schreier's Lemma.

<div class="alert alert-success">
<details>
<summary><b>Proof.</b></summary>

Recall that $t_y \in G$ satisfies $x.t_y = y$ for each $y \in x^G$, with $t_x = 1$.

**The Schreier generators lie in $G_x$.**  For any $y \in x^G$ and $a \in A$, let $z = y.a \in x^G$.  Then
$$
x.(t_y \, a \, t_z^{-1})
= (x.t_y).a.t_z^{-1}
= y.a.t_z^{-1}
= z.t_z^{-1} = x,
$$
so $t_y \, a \, t_z^{-1} \in G_x$.

**The Schreier generators generate $G_x$.**  Let $g \in G_x$.  Since $G$ is finite and $G = \langle A \rangle$, we can write $g = a_1 a_2 \cdots a_k$ with each $a_i \in A$ (no inverses needed).  Define
$$
y_0 = x, \qquad y_i = x.(a_1 \cdots a_i) \quad (1 \le i \le k),
$$
so that $y_k = x.g = x$.  Setting $s_i = t_{y_{i-1}} \, a_i \, t_{y_i}^{-1}$, the product telescopes:
$$
s_1 s_2 \cdots s_k
= t_{y_0} \, a_1 \underbrace{t_{y_1}^{-1} t_{y_1}}_{1} a_2 \cdots \underbrace{t_{y_{k-1}}^{-1} t_{y_{k-1}}}_{1} a_k \, t_{y_k}^{-1}
= t_x \, g \, t_x^{-1} = g.
$$
Thus every $g \in G_x$ is a product of Schreier generators. $\blacksquare$

</details>
</div>

**Exercise 1.16.**  Show that the power set $2^A = \{B : B \subseteq A\}$ of a (finite) set $A$ is the orbit of $A$ under the take-away action:
$$
B.S = B \setminus S = \{b \in B : b \notin S\}, \quad B, S \subseteq A.
$$

In [ ]:
using OrbitAl.syt: takeAway
set = Set("pizza")
orbit(set, set, takeAway)

**Exercise 1.17.**  Describe other interesting problems, where the right choice of action turns the solution into an application of the orbit algorithm.

**Exercise 1.18.**  A **composition** of $n$ is a sequence $c = (c_1, \dots, c_k)$ of positive integers whose sum is $n$.  Set up a bijection between the set of compositions of $n$ and the power set of $A = \{1,\dots, n{-}1\}$ (so that $A$ corresponds to the composition $(n)$, and the subset relationship on $2^A$ corresponds to refinement of compositions).  Use the bijective correspondence to compute the set of all compositions of $n$ as orbit of $(n)$ under a suitable take-away action.

**Exercise 1.19.**  A **partition** of $n$ is a composition $\lambda = (l_1, \dots, l_k)$ where $l_1 \geq \dots \geq l_k$.  Thus, sorting the parts of any composition in decreasing order yields a partition.  In that sense, a partition is a canonical representative of a rearrangement class of compositions.  Compute the partitions of $n$ as orbit of the partition $(n)$ under a suitable action on canonical composition rearrangement class representatives.

In [ ]:
function partitions(n)
    # your code here: return the list of all partitions of n, each as a decreasing vector
end

In [ ]:
import OrbitAl
mine = partitions(6)
theirs = OrbitAl.syt.partitions(6)
!isnothing(mine) && sort(mine) == sort(theirs)

**Exercise 1.20.**  Formulate a version of BFS that, for a given vertex $x$ in a simple connected graph $\Gamma$, finds a **shortest path** to any vertex $y$ in $\Gamma$.

**Exercise 1.21.**  Formulate an algorithm that, for given vertex in a simple connected graph $\Gamma$, finds **all** shortest paths to any vertex $y$ in $\Gamma$.

**Exercise 1.22.**  Say that a partition $\lambda$ **covers** a partition $\mu$ if $\mu$ can be obtained by decreasing a part of $\lambda$ by $1$.  Denote by $\geq$ the reflexive and transitive closure of the covering relation.  The **Young lattice** $Y(\lambda)$ is the graph on all partitions $\mu \leq \lambda$ with the covering relation as edges.  For a partition $\lambda$, compute its Young lattice $Y(\lambda)$ as orbit of $\lambda$ under a suitable action.

**Exercise 1.23.**  A **standard Young tableau** (SYT) of shape $\lambda$ is a shortest path from the empty partition $\emptyset$ to $\lambda$ in the Young lattice $Y(\lambda)$. For a given partition $\lambda$, compute all SYTs of shape $\lambda$ as a set of shortest paths.

In [ ]:
function standardYTs(lambda)
    # your code here: return the list of all SYTs of shape lambda
end

In [ ]:
import OrbitAl
lambdas = OrbitAl.syt.partitions(5)
mine = [standardYTs(l) for l in lambdas]
theirs = [OrbitAl.syt.standardYTs(l) for l in lambdas]
all(!isnothing(m) && length(m) == length(t) for (m, t) in zip(mine, theirs))

**Exercise 1.24.**  A **round trip** of shape $\lambda$ is a shortest path from $\emptyset$ to $\lambda$ and back along a 
(possibly different) shortest path. Verify that the total number of round trips to all the partitions $\lambda$ of $4$ is $24$.  What is the general formula for the total number of round trips for all partitions of $n$? Why?

In [ ]:
import OrbitAl
sum(length(OrbitAl.syt.standardYTs(x))^2 for x in OrbitAl.syt.partitions(4))

**Exercise 1.25** (Words from the Chain).  Along the recursion of `memberOfGroup`, keep track of words in the generators of $G$: each transversal element $t_y$ comes with a word $w_y$, as in `orbit_with_words`, and each Schreier generator $t_y a t_{y.a}^{-1}$ with the word $w_y\, w_a\, w_{y.a}^{-1}$, where $w_a$ is the word of the generator $a$, and a letter $-k$ stands for the inverse of the generator $k$.  Write a function `wordOfElement(group, a)` that returns a word for $a$ in the generators of `group`, or `nothing` if $a \notin G$.  For the symmetric groups generated by the swaps, compare the lengths of these words with the lengths of shortest words.  What happens for $M_{24}$, and why?  (With such words, a homomorphism defined on the generators can be evaluated anywhere: $\phi(a) = \phi(s_1) \dotsm \phi(s_k)$.)

In [ ]:
function wordOfElement(group::PermGp, a::Perm)
    # your code here: return a word for a in the generators of group, with -k for the inverse of generator k, or nothing if a is not in group
end

In [ ]:
function check_words()
    evaluate(G, w) = prod((k > 0 ? G.gens[k] : inv(G.gens[-k]) for k in w); init = G.one)
    S6 = PermGp(transpositions(6), Perm(6))
    isnothing(wordOfElement(S6, S6.one)) && return false
    all(evaluate(S6, wordOfElement(S6, a)) == a for a in elements(S6)) || return false
    D10 = PermGp([Perm([2, 3, 4, 5, 1]), Perm([1, 5, 4, 3, 2])], Perm(5))    # dihedral, of order 10
    S5 = PermGp(transpositions(5), Perm(5))
    all(isnothing(wordOfElement(D10, a)) == !(a in D10) for a in elements(S5))
end
check_words()

<div class="alert alert-success">
<details>
<summary><b>Answer.</b></summary>

**The recursion.**  As in `memberOfGroup`, $a = (a/t_y)\, t_y$ with $y = x.a$, so the word of $a$ is the word of $a/t_y$ in the stabilizer, followed by $w_y$.  The generators of the stabilizer are the distinct Schreier generators, as in `memberOfGroup`, each with the first word found for it, and the recursion carries these words along.

**Symmetric groups.**  For $S_n$, generated by the swaps $(i, i{+}1)$, every word found is a shortest one, for $n \leq 7$ (see the code below).  Compare Part 2: $t_y$ is the shortest element in its coset of the stabilizer $S_{n-1}$, a parabolic subgroup, and lengths add, $\ell(a) = \ell(a/t_y) + \ell(t_y)$.

**$M_{24}$.**  The words have hundreds or thousands of letters.  The word $w_y\, w_a\, w_{y.a}^{-1}$ of a Schreier generator is about twice as long as the words of the transversal, and it becomes a single letter at the next link.  So the longest word of a Schreier generator grows from $14$ letters at the first link to $60$, $168$, $289$, $474$ and $830$ at the next ones.  For the symmetric groups, in contrast, the swaps of $S_{n-1}$ are themselves Schreier generators, with words of length $1$.

</details>
</div>


In [ ]:
inverse(w) = -reverse(w)

# orbit_with_stabilizer, with words: for the transversal, and for the distinct Schreier generators
function orbit_with_stabilizer_words(aaa, words, x)
    list = [x]
    index = Dict(x => 1)
    reps = [aaa[1]^0]
    reps_words = [Int[]]
    stab = Dict{Perm, Vector{Int}}()    # the Schreier generators, with their words
    for (i, y) in enumerate(list)
        for (k, a) in enumerate(aaa)
            z = y^a
            t = reps[i] * a
            w = [reps_words[i]; words[k]]
            l = get!(index, z) do
                push!(list, z)
                push!(reps, t)
                push!(reps_words, w)
                length(list)
            end
            t == reps[l] || get!(stab, t / reps[l], [w; inverse(reps_words[l])])
        end
    end
    return (list = list, index = index, reps = reps, words = reps_words, stab = stab)
end

# words: the words of the generators of group, in the original generators
function wordOfElement(group::PermGp, a::Perm, words = [[k] for k in eachindex(group.gens)])
    is_trivial(group) && return a == group.one ? Int[] : nothing
    x = last_moved(group)
    orb = orbit_with_stabilizer_words(group.gens, words, x)
    l = get(orb.index, x^a, 0)
    l == 0 && return nothing
    stab = PermGp(collect(keys(orb.stab)), group.one)
    w = wordOfElement(stab, a / orb.reps[l], collect(values(orb.stab)))
    isnothing(w) && return nothing
    return [w; orb.words[l]]
end

# shortest words in S7, by orbit_with_words
S7 = PermGp(transpositions(7), Perm(7))
o = orbit_with_words(S7.gens, S7.one, onRight)
shortest = all(length(wordOfElement(S7, a)) == length(w) for (a, w) in zip(o.list, o.words))

check_words(), shortest, [length(wordOfElement(m24, randomGroupElement(m24))) for _ in 1:3]


**Exercise 1.26.**  ($*$) In practice, the number of Schreier generators of the stabilizers in the chain can grow very fast, in larger examples.  The **Schreier-Sims** algorithm intertwines orbit calculations and membership tests to keep the number of necessary generators small.  Implement such a strategy.  (Part 5 develops the Schreier-Sims algorithm.)

**Exercise 1.27.**  Describe an algorithm that uses BFS to compute the intersection $G \cap H$ of two permutation groups $G$ and $H$.

<div class="alert alert-success">
<details>
<summary><b>Answer.</b></summary>

$G$ acts on the right cosets $Hx$ of $H$ in the symmetric group by right multiplication, $Hx.a = Hxa$, and $Ha = H$ if and only if $a \in H$.  So $G \cap H$ is the stabilizer of the point $H$ in this action, and BFS from $H$, as in `orbit_with_stabilizer`, finds Schreier generators of $G \cap H$ (Schreier's Lemma).  The orbit has length $|G : G \cap H|$.

The orbit algorithm needs to recognize a coset that it has seen before, so each coset needs a canonical form: here, the sorted list of its elements.  This lists all of $H$, once for each coset in the orbit.  Part 5 does better: a backtrack search through $G$, which prunes with a stabilizer chain of $H$ (see *Intersections* there).

</details>
</div>

In [ ]:
# the right coset H x, as the sorted list of its elements, under right multiplication
onCosets(c, a) = sort([h * a for h in c])

# G ∩ H is the stabilizer of the coset H, in the action of G on the right cosets of H
function intersection_bfs(G::PermGp, H::PermGp)
    orb = orbit_with_stabilizer(G.gens, elements(H), onCosets)
    return (group = PermGp(unique(orb.stab), G.one), index = length(orb.list))
end

a, b = Perm([2,3,1,4]), Perm([2,1,4,3])     # A4
r, s = Perm([2,3,4,1]), Perm([3,2,1,4])     # D8
K = intersection_bfs(PermGp([a, b], Perm(4)), PermGp([r, s], Perm(4)))
elements(K.group), K.index                  # V4, of index 3 in A4

**Exercise 1.28** (Faster Conjugacy Classes).  Improve `conjClasses` in two ways.  First, compute each class by an orbit algorithm that keeps track of the elements seen so far in a `Set`, rather than searching through the list (compare the dictionary `index` in `orbit_with_stabilizer`).  Second, in the orbit algorithm on classes, look up the product $xa$ in the classes found so far, and compute its class only if it is new.  Write a function `conjClasses2(gp)` that returns the list of classes, each as a set of elements, and compare its running times with those of `conjClasses` on the symmetric groups of degree $6$, $7$ and $8$.

In [ ]:
function conjClasses2(gp::PermGp)
    # your code here: return the list of conjugacy classes, each as a Set of elements
end

In [ ]:
G = PermGp(transpositions(6), Perm(6))
mine = conjClasses2(G)
!isnothing(mine) && sort(length.(mine)) == sort(size.(conjClasses(G)))

In [ ]:
function conjClasses2(gp::PermGp)
    # a class as a Set, by the orbit algorithm with a set of the elements seen so far
    function class_set(x)
        seen = Set([x])
        list = [x]
        for y in list, a in gp.gens
            z = y^a
            z in seen || begin
                push!(seen, z)
                push!(list, z)
            end
        end
        return seen
    end
    # the orbit algorithm on classes: look up x * a before computing its class
    gens = orbitx(gp.gens, copy(gp.gens), onPoints)
    reps = [gp.one]
    classes = [Set([gp.one])]
    for x in reps, a in gens
        z = x * a
        any(c -> z in c, classes) && continue
        push!(reps, z)
        push!(classes, class_set(z))
    end
    return classes
end

G = PermGp(transpositions(7), Perm(7))
t1 = @elapsed c1 = conjClasses(G)
t2 = @elapsed c2 = conjClasses2(G)
sort(length.(c2)) == sort(size.(c1)), round(t1, digits = 2), round(t2, digits = 3)